In [ ]:
import os
import base64
from pathlib import Path
from email.mime.text import MIMEText

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/gmail.send"]
TOKEN_FILE = Path("gmail_token.json")

client_config = {
    "installed": {
        "client_id": os.environ["fallmailclient_id"],
        "client_secret": os.environ["fallmailclient_pass"],
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "redirect_uris": ["http://localhost"],
    }
}

creds = None
if TOKEN_FILE.exists():
    creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_config(client_config, SCOPES)
        creds = flow.run_local_server(port=0, open_browser=True)

    TOKEN_FILE.write_text(creds.to_json(), encoding="utf-8")

gmail = build("gmail", "v1", credentials=creds)

recipient = "kamesh29kumar@gmail.com"
message = MIMEText("Hi I am K.A.R.U.P and this is a test email")
message["To"] = recipient
message["Subject"] = "Gmail API test"

response = gmail.users().messages().send(
    userId="me",
    body={"raw": base64.urlsafe_b64encode(message.as_bytes()).decode()},
).execute()

print(f"Sent successfully — message ID: {response['id']}")

Sent successfully — message ID: 1a03e05aaea29087


In [4]:
# !pip install -q google-api-python-client google-auth-oauthlib

import os
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

BST = ZoneInfo("Europe/London")
TOKEN_FILE = Path("google_token.json")

# Required to read event names/details, not merely free/busy status.
SCOPES = ["https://www.googleapis.com/auth/calendar.events.readonly"]

client_config = {
    "installed": {
        "client_id": os.environ["fallmailclient_id"],
        "client_secret": os.environ["fallmailclient_pass"],
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "redirect_uris": ["http://localhost"],
    }
}

creds = None
if TOKEN_FILE.exists():
    creds = Credentials.from_authorized_user_file(TOKEN_FILE)

# Re-authorize if this token lacks event-reading access.
if not creds or not creds.valid or not creds.has_scopes(SCOPES):
    flow = InstalledAppFlow.from_client_config(client_config, SCOPES)
    creds = flow.run_local_server(port=0, open_browser=True)
    TOKEN_FILE.write_text(creds.to_json(), encoding="utf-8")

calendar = build("calendar", "v3", credentials=creds)

# This week's range: Monday 00:00 BST → next Monday 00:00 BST
now = datetime.now(BST)
week_start = (now - timedelta(days=now.weekday())).replace(
    hour=0, minute=0, second=0, microsecond=0
)
week_end = week_start + timedelta(days=7)

events = []
page_token = None

while True:
    result = calendar.events().list(
        calendarId="primary",
        timeMin=week_start.isoformat(),
        timeMax=week_end.isoformat(),
        timeZone="Europe/London",
        singleEvents=True,
        orderBy="startTime",
        maxResults=2500,
        pageToken=page_token,
    ).execute()

    events.extend(result.get("items", []))
    page_token = result.get("nextPageToken")
    if not page_token:
        break

print(f"Calendar: {week_start:%A %d %B %Y} — {(week_end - timedelta(days=1)):%A %d %B %Y} (BST)\n")

if not events:
    print("No events this week.")
else:
    for event in events:
        start = event["start"].get("dateTime") or event["start"].get("date")
        end = event["end"].get("dateTime") or event["end"].get("date")
        title = event.get("summary", "(No title)")
        print(f"{start} → {end} | {title}")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=898018817631-05o5tdcb5rpig179jd3hck77kt63q2nv.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A55899%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.events.readonly&state=g3jnxRJp9PW7xSHWwoPrK8Su91K7DD&code_challenge=XQ-eP6PooRLj3VcNl8qL6GlyllbpPC-oBEc6kiX4EOo&code_challenge_method=S256&access_type=offline


gio: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=898018817631-05o5tdcb5rpig179jd3hck77kt63q2nv.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A55899%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.events.readonly&state=g3jnxRJp9PW7xSHWwoPrK8Su91K7DD&code_challenge=XQ-eP6PooRLj3VcNl8qL6GlyllbpPC-oBEc6kiX4EOo&code_challenge_method=S256&access_type=offline: Operation not supported


Calendar: Monday 31 August 2026 — Sunday 06 September 2026 (BST)

2026-09-04T12:00:00+01:00 → 2026-09-04T13:00:00+01:00 | Interview with manish
